In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pywt

In [8]:
IN_PATH  = "../../data/NF-UNSW-NB15-v3.csv"
wavelet_name = 'cmor3.5-1.0'
max_samples = 150
scales = np.arange(1, 128)
random_seed = 42

In [5]:
df = pd.read_csv(IN_PATH)

In [17]:
df['flow_id'] = df.index

In [18]:
attack_types = df['Attack'].value_counts()
print("Flows per attack type:\n", attack_types)

Flows per attack type:
 Attack
Benign            2237731
Exploits            42748
Fuzzers             33816
Generic             19651
Reconnaissance      17074
DoS                  5980
Backdoor             4659
Shellcode            2381
Analysis             1226
Worms                 158
Name: count, dtype: int64


In [ ]:
# The effective sample per attack type is the smaller of max_samples or actual flows
sample_sizes = {attack: min(count, max_samples) for attack, count in attack_types.items()}
print("Sample sizes per attack type:\n", sample_sizes)

Sample sizes per attack type:
 {'Benign': 150, 'Exploits': 150, 'Fuzzers': 150, 'Generic': 150, 'Reconnaissance': 150, 'DoS': 150, 'Backdoor': 150, 'Shellcode': 150, 'Analysis': 150, 'Worms': 150}


In [13]:
IAT_FEATURES = [
    "SRC_TO_DST_IAT_MIN",
    "SRC_TO_DST_IAT_AVG",
    "SRC_TO_DST_IAT_MAX",
    "SRC_TO_DST_IAT_STDDEV",
    "DST_TO_SRC_IAT_MIN",
    "DST_TO_SRC_IAT_AVG",
    "DST_TO_SRC_IAT_MAX",
    "DST_TO_SRC_IAT_STDDEV"
]

In [11]:
rows = []

In [19]:
for attack_type in attack_types.index:
    df_attack = df[df['Attack'] == attack_type].sample(
        n=min(len(df[df['Attack'] == attack_type]), max_samples),
        random_state=random_seed
    )
    
    for _, row in df_attack.iterrows():
        signal = row[IAT_FEATURES].values.astype(float)
        signal = np.nan_to_num(signal, nan=0.0)
        
        if np.std(signal) == 0:
            continue

        signal = (signal - signal.mean()) / signal.std()
        

        coeffs, _ = pywt.cwt(signal, scales, wavelet_name)
        coeff_mag = np.abs(coeffs)
        
        features = coeff_mag.mean(axis=1)
        features = (features - features.mean()) / features.std()
        
        out = {"attack_type": attack_type, "flow_id": row["flow_id"]}
        for i, val in enumerate(features):
            out[f"cwt_scale_{i}"] = val
        
        rows.append(out)

In [20]:
OUT_PATH = "cwt_150_multiclass_attack.csv"

In [21]:
df_cwt_binary = pd.DataFrame(rows)
df_cwt_binary.to_csv(OUT_PATH, index=False)
df_cwt_binary.head()

,attack_type,flow_id,cwt_scale_0,cwt_scale_1,cwt_scale_2,cwt_scale_3,cwt_scale_4,cwt_scale_5,cwt_scale_6,cwt_scale_7,...,cwt_scale_117,cwt_scale_118,cwt_scale_119,cwt_scale_120,cwt_scale_121,cwt_scale_122,cwt_scale_123,cwt_scale_124,cwt_scale_125,cwt_scale_126
0,Benign,924483,-0.743641,3.605484,3.560675,7.368320,4.713295,1.789026,0.193087,0.210379,...,0.003726,0.008305,0.012864,0.017405,0.021927,0.026430,0.030916,0.035383,0.039832,0.044264
1,Benign,1991972,-0.551598,3.172561,3.802309,7.592432,5.234969,2.414897,0.476657,-0.646720,...,-0.032361,-0.028692,-0.025037,-0.021398,-0.017773,-0.014164,-0.010569,-0.006989,-0.003422,0.000130
2,Benign,1595752,-0.664497,0.663171,3.162886,7.477042,5.324979,2.265312,0.201699,-0.003630,...,-0.348184,-0.345337,-0.342502,-0.339678,-0.336867,-0.334067,-0.331278,-0.328500,-0.325734,-0.322978
3,Benign,1438936,-0.551654,4.884650,3.997877,7.146777,4.461126,1.826211,0.172284,-0.716152,...,0.201821,0.206342,0.210844,0.215327,0.219792,0.224238,0.228667,0.233077,0.237470,0.241846
4,Benign,598588,-0.551654,4.884650,3.997877,7.146777,4.461126,1.826211,0.172284,-0.716152,...,0.201821,0.206342,0.210844,0.215327,0.219792,0.224238,0.228667,0.233077,0.237470,0.241846
